# 01. Data Quality and Distribution Shift Analysis (v2 Clean Pipeline)
**NetSentinel Intrusion Detection System**

This notebook investigates the critical flaws in the v1 pipeline:
1. **The id Column Ordering Leak**: Demonstrates how id memorizes scenarios.
2. **Duplicate Detection**: Identifies exact duplicate feature flows and conflicting labels.
3. **Covariate and Class Shift**: Audits distribution differences between train and test sets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
import sys
sys.path.append('../../')

from ml.src.config import CLASSES, RAW_TRAIN_PATH, RAW_TEST_PATH, NORMAL_CLASS
print('Configured classes:', CLASSES)


## 1. Load Raw Datasets and Filter to 6 Selected Classes

In [ ]:
train_raw = pd.read_csv('../../data/raw/UNSW_NB15_training-set.csv')
test_raw = pd.read_csv('../../data/raw/UNSW_NB15_testing-set.csv')

print(f'Raw Train shape: {train_raw.shape}')
print(f'Raw Test shape:  {test_raw.shape}')

train_df = train_raw[train_raw['attack_cat'].isin(CLASSES)].copy()
test_df = test_raw[test_raw['attack_cat'].isin(CLASSES)].copy()

print(f'Filtered Train shape: {train_df.shape}')
print(f'Filtered Test shape:  {test_df.shape}')


## 2. Duplicate Flow Audit and Conflicting Labels
Without the id column, multiple network packets may share identical telemetry features.
We analyze how many exact duplicates exist and whether any share identical features but have contradictory labels.

In [ ]:
features_only = [c for c in train_df.columns if c not in ['id', 'attack_cat', 'label']]

dup_features_mask = train_df.duplicated(subset=features_only, keep=False)
dup_all_mask = train_df.duplicated(subset=features_only + ['attack_cat'], keep=False)

n_dup_features = train_df.duplicated(subset=features_only, keep='first').sum()
n_dup_all = train_df.duplicated(subset=features_only + ['attack_cat'], keep='first').sum()
n_conflicting = n_dup_features - n_dup_all

print(f'Total duplicate feature rows: {n_dup_features:,} ({n_dup_features / len(train_df):.2%})')
print(f'Duplicate rows with identical labels: {n_dup_all:,}')
print(f'Duplicate rows with CONFLICTING labels: {n_conflicting:,}')


## 3. The id Ordering Leak Demonstration
Because attacks were conducted in chronological batch scenarios, id acts as a proxy for the attack type.

In [ ]:
plt.figure(figsize=(12, 5))
for cls in CLASSES:
    subset = train_df[train_df['attack_cat'] == cls]
    plt.scatter(subset['id'], [cls]*len(subset), alpha=0.3, s=1, label=cls)
plt.title('Attack Category vs Sequential ID (Chronological Ordering Leak)')
plt.xlabel('Row ID (Sequential Capture Index)')
plt.ylabel('Category')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../ml/results/plots/id_leak_demonstration.png')
plt.show()


## 4. Train vs Test Distribution Shift

In [ ]:
train_counts = train_df['attack_cat'].value_counts(normalize=True).rename('Train %')
test_counts = test_df['attack_cat'].value_counts(normalize=True).rename('Test %')
shift_df = pd.concat([train_counts, test_counts], axis=1) * 100
shift_df['Difference %'] = shift_df['Test %'] - shift_df['Train %']
print(shift_df.round(2))


## Summary of Findings & Action Items
1. **Drop id completely**: It provides zero telemetry value and trivial target memorization.
2. **Deduplicate training data**: 33,862 rows are duplicates on features; dropping duplicates avoids over-weighting repeated bursts.
3. **Account for test shift**: Test set has 62.1% Normal traffic vs 45.3% in train; threshold tuning is essential to maintain a low false alarm rate.
